In [2]:
import pandas as pd
import numpy as np

from LSTM.Clustering_LSTM import training_clustered

In [3]:
training = pd.read_csv("C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/training_clustered.csv").drop(columns=['Unnamed: 0'])
val = pd.read_csv("C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/validation.csv").drop(columns=['Unnamed: 0'])

Val_X_df = pd.read_csv(f"C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/Val_X_clustered.csv").drop(columns=['Unnamed: 0'])
Val_y_df = pd.read_csv("C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/Val_y.csv").drop(columns=['Unnamed: 0'])

# Normalize LAT and LON

In [4]:
from pyproj import Transformer
tf = Transformer.from_crs("EPSG:4326", "EPSG:32635", always_xy=True)  # zone 35N covers 24-30°E
x, y = tf.transform(training['LON'].values, training['LAT'].values)
training['LON_meters'] = x
training['LAT_meters'] = y

# validation trajectories X and y with assigned clusters
Val_X_x, Val_X_y = tf.transform(Val_X_df['LON'].values, Val_X_df['LAT'].values)
Val_X_df["LON_meters"] = Val_X_x
Val_X_df["LAT_meters"] = Val_X_y

Val_Y_x, Val_Y_y = tf.transform(Val_y_df['LON'].values, Val_y_df['LAT'].values)
Val_y_df["LON_meters"] = Val_Y_x
Val_y_df["LAT_meters"] = Val_Y_y

In [5]:
from sklearn.preprocessing import MinMaxScaler
xy_scaler = MinMaxScaler()

# Fit on only training data
training[["X","Y"]] = xy_scaler.fit_transform(training[["LON_meters","LAT_meters"]])


In [6]:
Val_X_df[["X","Y"]] = xy_scaler.transform(Val_X_df[["LON_meters","LAT_meters"]])
Val_y_df[["X","Y"]] = xy_scaler.transform(Val_y_df[["LON_meters","LAT_meters"]])

# Normalize SOG

In [7]:
Val_X_df

,index,LON,LAT,SOG,COG,x_m,y_m,assigned_cluster_x,nearest_trajectory_ID_x,assigned_cluster_y,nearest_trajectory_ID_y,assigned_cluster,nearest_trajectory_ID,LON_meters,LAT_meters,X,Y
0,0,24.954517,37.437150,NaN,NaN,319038.078864,4.145333e+06,-1,11850,-1,11850,-1,11850,319038.048907,4.145333e+06,0.125760,0.698240
1,0,24.956266,37.438070,5.996785,56.586765,319195.044855,4.145431e+06,-1,11850,-1,11850,-1,11850,319195.013057,4.145431e+06,0.126443,0.698886
2,0,24.958200,37.438934,6.350474,60.551440,319368.247955,4.145524e+06,-1,11850,-1,11850,-1,11850,319368.206939,4.145524e+06,0.127196,0.699490
3,0,24.960367,37.439632,6.690756,67.857990,319561.631868,4.145597e+06,-1,11850,-1,11850,-1,11850,319561.612930,4.145597e+06,0.128038,0.699970
4,0,24.962767,37.440100,7.067566,76.238655,319775.050812,4.145644e+06,-1,11850,-1,11850,-1,11850,319775.078118,4.145644e+06,0.128966,0.700279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40315,1343,26.417910,37.208160,9.357062,306.919370,448349.174630,4.118123e+06,2,18621,2,18621,2,18621,448349.207882,4.118123e+06,0.688192,0.520130
40316,1343,26.415295,37.209675,9.277826,306.013500,448118.173837,4.118293e+06,2,18621,2,18621,2,18621,448118.205253,4.118293e+06,0.687187,0.521239
40317,1343,26.412695,37.211150,9.158244,305.494450,447888.510013,4.118458e+06,2,18621,2,18621,2,18621,447888.515361,4.118458e+06,0.686188,0.522320
40318,1343,26.410076,37.212624,9.201447,305.246950,447657.160178,4.118623e+06,2,18621,2,18621,2,18621,447657.147915,4.118623e+06,0.685182,0.523400


In [8]:
SOG_scaler = MinMaxScaler()  # It expects a 2D array
# Fit on only training data
training[["SOG"]] = SOG_scaler.fit_transform(training[["SOG"]])
Val_X_df[["SOG"]] = SOG_scaler.transform(Val_X_df[["SOG"]])

# Normalize COG

In [9]:
# Convert COG from degrees into radians, then encode it
for df in [training, Val_X_df]:
    df["COG_sin"] = np.sin(np.deg2rad(df["COG"]))
    df["COG_cos"] = np.cos(np.deg2rad(df["COG"]))

In [10]:
Val_y_df

,index,LON,LAT,LON_meters,LAT_meters,X,Y
0,0,25.011267,37.414818,324006.914503,4.142747e+06,0.147372,0.681316
1,0,25.011883,37.413883,324059.245149,4.142642e+06,0.147600,0.680630
2,0,25.012500,37.413000,324111.787350,4.142543e+06,0.147828,0.679981
3,0,25.013100,37.412132,324162.861322,4.142446e+06,0.148050,0.679343
4,0,25.013800,37.411400,324223.105358,4.142363e+06,0.148312,0.678803
...,...,...,...,...,...,...,...
13435,1343,26.392042,37.222717,446064.195289,4.119753e+06,0.678254,0.530796
13436,1343,26.389488,37.224186,445838.661956,4.119917e+06,0.677273,0.531872
13437,1343,26.386969,37.225605,445616.206498,4.120076e+06,0.676305,0.532912
13438,1343,26.384417,37.227062,445390.859320,4.120239e+06,0.675325,0.533979


In [13]:
for df, name in [(training, "training_clustered")]:
    df.to_csv(f"C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/{name}_normalized.csv")

In [12]:
Val_X_df.to_csv(f"C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/Val_X_normalized_cluster.csv")
Val_y_df.to_csv(f"C:/Users/chene/Documents/MasterAbschlussArbeit/Data/LSTM_2_try/Val_y_normalized.csv")